# 📚 Popular Online Scraper — v2 (Detail Page)

Uses `curl_cffi` to impersonate a real Chrome TLS fingerprint and bypass Cloudflare.

**What's new in v2:**
- Visits each book's individual page to extract **ISBN, Publisher, Format, Language, Description** and more
- All extra fields saved to Excel

**Workflow:**
1. Run all cells in order
2. Results saved to `Popular_Books_Data.xlsx`


In [1]:
%pip install curl-cffi beautifulsoup4 pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from curl_cffi import requests as cf_requests
from bs4 import BeautifulSoup
import time, random, re, pandas as pd, os

print("✅ All imports successful!")

✅ All imports successful!


## 2. Configure Search URLs

Edit the `urls` list below. The scraper auto-detects total pages and loops through all of them.

In [3]:
# ── Popular Online category URLs to scrape ──────────────────────────────────
# Find these by browsing the site and copying the URL of any category/search page.
# The scraper will auto-detect pagination and loop all pages.

urls = [
    "https://www.popularonline.com.my/default/catalog/category/view/s/Business%20&%20Management/id/5899/?did=5897",
    "https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Robin%20Graded%20Readers/id/5900/?did=5897",
    "https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Pre%20School/id/5901/?did=5897"
    "https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Books/id/5902/?did=5897"    
]

# Delay range between page requests (seconds) — keep this generous to avoid blocks
MIN_DELAY = 4
MAX_DELAY = 9

print(f"🔗 {len(urls)} URL(s) queued.")
for u in urls:
    print(f"   {u}")

🔗 3 URL(s) queued.
   https://www.popularonline.com.my/default/catalog/category/view/s/Business%20&%20Management/id/5899/?did=5897
   https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Robin%20Graded%20Readers/id/5900/?did=5897
   https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Pre%20School/id/5901/?did=5897https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Books/id/5902/?did=5897


## 3. Helper — Shared Session

`curl_cffi` impersonates Chrome's TLS fingerprint. We reuse one session across all requests.

In [4]:
import asyncio

CONCURRENCY = 5  # max simultaneous detail-page requests

session = cf_requests.AsyncSession(impersonate="chrome120")

HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "accept-language": "en-MY,en;q=0.9,ms;q=0.8",
    "referer": "https://www.popularonline.com.my/",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

async def fetch_page(url, retries=3):
    """Async fetch with retries. Returns BeautifulSoup or None."""
    for attempt in range(1, retries + 1):
        try:
            resp = await session.get(url, headers=HEADERS, timeout=20)
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "html.parser")
            elif resp.status_code == 403:
                print(f"⚠️  403 Cloudflare block on attempt {attempt}/{retries}.")
                await asyncio.sleep(random.uniform(10, 20))
            elif resp.status_code == 429:
                print(f"⚠️  429 Rate limited on attempt {attempt}/{retries}. Waiting 30s...")
                await asyncio.sleep(30)
            else:
                print(f"⚠️  HTTP {resp.status_code} on attempt {attempt}/{retries}.")
                await asyncio.sleep(5)
        except Exception as e:
            print(f"❌ Request error on attempt {attempt}/{retries}: {e}")
            await asyncio.sleep(5)
    return None

print("🔌 Testing connection to Popular Online...")
test_soup = await fetch_page("https://www.popularonline.com.my/")
if test_soup:
    print("✅ Connected successfully! Cloudflare bypassed.")
else:
    print("❌ Could not connect. Cloudflare may be blocking curl_cffi too.")
    print("   → Try running this on a different network or VPN.")


🔌 Testing connection to Popular Online...
✅ Connected successfully! Cloudflare bypassed.


## 4. Parse Helpers

`parse_products()` collects the listing-level fields.  
`parse_detail_page()` visits each book URL and extracts the extra fields.  
If the site redesigns, only these need updating.

In [5]:
def get_total_pages(soup):
    """Detect total pages from Popular Online pagination."""
    try:
        amount = soup.select_one('.amount')
        if amount:
            match = re.search(r'of (\d+)', amount.get_text())
            if match:
                total_items = int(match.group(1))
                items_per_page = 180
                return -(-total_items // items_per_page)
        pages = soup.select('.pages ol li a')
        nums = [int(p.get_text(strip=True)) for p in pages if p.get_text(strip=True).isdigit()]
        return max(nums) if nums else 1
    except Exception:
        return 1


def parse_products(soup):
    """Extract listing-level rows (name, price, author, URL)."""
    items = []
    for product in soup.select('li.item.col-sm-3'):
        try:
            name_el = product.select_one('h2.product-name a')
            name = name_el.get_text(strip=True) if name_el else 'N/A'

            price = 'N/A'
            member_prices = product.select('p.member-price')
            if member_prices:
                price_spans = member_prices[0].select('span.price span')
                if len(price_spans) >= 2:
                    price = price_spans[1].get_text(strip=True)

            usual_price = 'N/A'
            usual_el = product.select_one('span.regular-price span.price')
            if usual_el:
                usual_price = usual_el.get_text(strip=True).replace('RM', '').strip()

            author_el = product.select_one('span.auth')
            author = author_el.get_text(strip=True).replace('Author:', '').strip() if author_el else 'N/A'

            url_el = product.select_one('h2.product-name a')
            url = url_el['href'] if url_el else 'N/A'

            items.append({
                'Product Name': name,
                'Usual Price (RM)': usual_price,
                'Now Price (RM)': price,
                'Author': author,
                'URL': url,
            })
        except Exception:
            continue
    return items


def parse_detail_page(soup):
    """
    Visit an individual book page and extract extra fields.
    Returns a dict — missing fields default to 'N/A'.
    """
    detail = {
        'ISBN': 'N/A',
        'Publisher': 'N/A',
        'Format': 'N/A',
        'Language': 'N/A',
        'Pages': 'N/A',
        'Publication Date': 'N/A',
        'Description': 'N/A',
        'Member Price (RM)': 'N/A',
        'Cover Image URL': 'N/A',
    }
    if soup is None:
        return detail

    try:
        # ── Description ──────────────────────────────────────────────────
        desc_el = (
            soup.select_one('#product-description .tab-content') or
            soup.select_one('#product-description')
        )
        if desc_el:
            detail['Description'] = desc_el.get_text(separator=' ', strip=True)[:1000]

        # ── Cover image ───────────────────────────────────────────────────
        img_el = (
            soup.select_one('img.etalage_source_image') or
            soup.select_one('img.etalage_thumb_image')
        )
        if img_el:
            detail['Cover Image URL'] = img_el.get('src', 'N/A')

        # ── Member price ──────────────────────────────────────────────────
        member_blocks = soup.select('p.member-price')
        if len(member_blocks) >= 2:
            spans = member_blocks[1].select('span.price span')
            if len(spans) >= 2:
                detail['Member Price (RM)'] = spans[1].get_text(strip=True)

        # ── Product attributes (Bootstrap grid layout) ────────────────────
        label_map = {
            'isbn/ean':         'ISBN',
            'isbn':             'ISBN',
            'isbn-10':          'ISBN',
            'isbn-13':          'ISBN',
            'publisher':        'Publisher',
            'format':           'Format',
            'binding type':     'Format',
            'binding':          'Format',
            'language':         'Language',
            'language ':        'Language',
            'pages':            'Pages',
            'no. of pages':     'Pages',
            'number of pages':  'Pages',
            'publication date': 'Publication Date',
            'pub date':         'Publication Date',
            'date added':       'Publication Date',
        }

        for item in soup.select('div.product-data-item'):
            label_el = item.select_one('div.product-data__label')
            value_el = item.select_one('div.product-data__value')
            if label_el and value_el:
                label = label_el.get_text(strip=True).lower().rstrip(':').strip()
                value = value_el.get_text(strip=True)
                if label in label_map and detail[label_map[label]] == 'N/A':
                    detail[label_map[label]] = value

    except Exception as e:
        print(f"  ⚠️  Detail parse error: {e}")

    return detail


## 5. Run Scraper (Listing + Detail Pages)

In [6]:
products = []

for base_url in urls:
    try:
        print(f"--- Processing: {base_url} ---")

        soup = await fetch_page(base_url)
        if soup is None:
            print("⏭️  Skipping — could not load page.")
            continue

        total_pages = min(get_total_pages(soup), 500)
        print(f"📄 Detected {total_pages} page(s).")

        # Page 1 (already fetched)
        page_items = parse_products(soup)
        products.extend(page_items)
        print(f"  ✅ Page 1/{total_pages}: {len(page_items)} items. (Total: {len(products)})")

        # Remaining listing pages (sequential)
        for page_num in range(2, total_pages + 1):
            sep = '&' if '?' in base_url else '?'
            page_url = f"{base_url}{sep}p={page_num}&limit=180"
            print(f"  🔍 Page {page_num}/{total_pages}...", end=" ")
            soup = await fetch_page(page_url)
            if soup is None:
                print("❌ Failed. Skipping.")
                continue
            page_items = parse_products(soup)
            products.extend(page_items)
            print(f"✅ {len(page_items)} items. (Total: {len(products)})")
            await asyncio.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    except Exception as e:
        print(f"❌ Critical error on {base_url}: {e}")

print(f"\n🏁 Listing scrape done — {len(products)} books collected.")
print(f"🔎 Now fetching detail pages concurrently (CONCURRENCY={CONCURRENCY})...")

sem = asyncio.Semaphore(CONCURRENCY)

async def fetch_detail(book, idx, total):
    url = book.get('URL', 'N/A')
    if url == 'N/A' or not url.startswith('http'):
        return
    async with sem:
        print(f"  [{idx}/{total}] {book['Product Name'][:55]}...", end=" ")
        detail_soup = await fetch_page(url)
        extra = parse_detail_page(detail_soup)
        book.update(extra)
        print(f"ISBN={extra['ISBN']}  Publisher={extra['Publisher']}")
        await asyncio.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

tasks = [fetch_detail(book, i, len(products)) for i, book in enumerate(products, 1)]
await asyncio.gather(*tasks)

print("\n✅ Detail scraping complete!")


--- Processing: https://www.popularonline.com.my/default/catalog/category/view/s/Business%20&%20Management/id/5899/?did=5897 ---
📄 Detected 4 page(s).
  ✅ Page 1/4: 60 items. (Total: 60)
  🔍 Page 2/4... ✅ 180 items. (Total: 240)
  🔍 Page 3/4... ✅ 180 items. (Total: 420)
  🔍 Page 4/4... ✅ 53 items. (Total: 473)
--- Processing: https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Robin%20Graded%20Readers/id/5900/?did=5897 ---
📄 Detected 1 page(s).
  ✅ Page 1/1: 63 items. (Total: 536)
--- Processing: https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Pre%20School/id/5901/?did=5897https://www.popularonline.com.my/default/catalog/category/view/s/Children%20-%20Books/id/5902/?did=5897 ---
📄 Detected 5 page(s).
  ✅ Page 1/5: 180 items. (Total: 716)
  🔍 Page 2/5... ✅ 180 items. (Total: 896)
  🔍 Page 3/5... ✅ 180 items. (Total: 1076)
  🔍 Page 4/5... ✅ 180 items. (Total: 1256)
  🔍 Page 5/5... ✅ 133 items. (Total: 1389)

🏁 Listing scrape don

## 6. Save to CSV

In [9]:
COLUMNS = [
    'Product Name', 'Author',
    'Usual Price (RM)', 'Now Price (RM)', 'Member Price (RM)',
    'ISBN', 'Publisher', 'Format', 'Language', 'Pages', 'Publication Date',
    'Description', 'Cover Image URL', 'URL',
]

if products:
    df = pd.DataFrame(products)
    for col in COLUMNS:
        if col not in df.columns:
            df[col] = 'N/A'
    df = df[COLUMNS]
    filename = 'Popular_Books_Data.csv'
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"✅ Scraped {len(df)} total items.")
    print(f"📁 Saved to: {os.path.abspath(filename)}")
else:
    print("⚠️  No data collected. Check the connection test in Cell 3.")


✅ Scraped 1389 total items.
📁 Saved to: c:\Users\Iman\Desktop\project_1\Popular_Books_Data.csv


## 7. Preview Results

In [10]:
if products:
    display(df.head(20))
else:
    print("No data to preview.")


,Product Name,Author,Usual Price (RM),Now Price (RM),Member Price (RM),ISBN,Publisher,Format,Language,Pages,Publication Date,Description,Cover Image URL,URL
0,THE 12-WEEK MBA,Bjorn Billhart,60.50,60.50,54.45,9781538781944,BALANCE,Paperback,English,384,2026/03/04,"A practical, fast-track guide to mastering cor...",https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
1,90 DAYS TO LEVEL UP YOUR LEADERSHIP,George,86.15,86.15,77.54,9781394257966,WILEY,Paperback,English,240,2026/05/05,Supercharge your leadership skills in just thr...,https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
2,THE SECRET LANGUAGE OF WORK,Michael Joseph,117.50,117.50,94.00,9780241802557,PENGUIN BOOKS LTD,Paperback,English,240,2026/04/10,Practical scripts and communication strategies...,https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
3,THE GAME OF THE IMPOSSIBLE,N/A,89.95,71.90,71.90,9789815351866,Penguin,Paperback,N/A,384,N/A,"""This book is about how to transform a company...",https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
4,5 VOICES: HOW TO COMMUNICATE EFFECTIVELY WITH ...,Kubicek,86.15,86.15,77.54,9781394369393,WILEY,Paperback,English,240,2026/05/05,Discover your leadership voice and unlock your...,https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
5,DRIVING PERFORMANCE: 10 LESSONS ABOUT BUILDING...,Goddard,114.95,114.95,103.46,9781394372942,WILEY,Paperback,English,256,2026/05/05,In Driving Performance: 10 Lessons About Build...,https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
6,Chip War: The Fight for the World's Most Criti...,Chris Miller,72.90,65.50,58.50,9781398557352,Simon & Schuster Ltd,Paperback,English,480,2026/04/10,Chris Miller explores how semiconductor chips ...,https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
7,The Wealth Habit: Small Changes that Will Make...,"Ken Okoroafor,Mary Okoroafor",114.90,114.90,103.41,9781529449204,Quercus,Paperback,English,384,2026/04/15,"In The Wealth Habit, Ken Okoroafor and Mary Ok...",https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
8,Why We're Getting Poorer: Economics for an Unf...,Cahal Moran,74.95,74.95,59.96,9780008637996,HarperCollins,Paperback,English,432,2026/04/03,"Examines how flawed economic policies, wage st...",https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
9,The Next Conversation Workbook,Jefferson Fisher,117.50,117.50,105.75,9780241805725,Penguin Life,Paperback,English,192,2026/04/15,A practical guide of exercises and prompts to ...,https://d26olvxuieoyaa.cloudfront.net/catalog/...,https://www.popularonline.com.my/default/catal...
